# Self-RAG Experimentation

1. **Hallucination Grader**
2. **Answer Quality Grader**
3. **Query Rewriter**

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from langchain_groq import ChatGroq
from src.config import SELF_RAG_MODEL, GROQ_API_KEY, TEMPERATURE, MAX_TOKENS

llm = ChatGroq(
    model=SELF_RAG_MODEL,
    api_key=GROQ_API_KEY,
    temperature=0,
    max_tokens=MAX_TOKENS
)
print("LLM Initialized!")

## 1. Hallucination Grader


In [ ]:
def hallucination_grader(context: str, answer: str):
    prompt = f"""
    You are a grader assessing whether an LLM generation is grounded in / supported by a set of retrieved facts.
    Give a binary score 'yes' or 'no'. 'Yes' means that the answer is grounded in and supported by the set of facts.
    
    Set of facts: {context}
    
    LLM generation: {answer}
    
    Binary Score ("yes" or "no"):
    """
    response = llm.invoke(prompt)
    return response.content.strip().lower()

context = "The Eiffel Tower is located in Paris, France. It was constructed in 1889."

In [ ]:
# Test 1: Grounded Answer
good_answer = "The Eiffel Tower was built in 1889 and is located in Paris."
print("Grounded Score:", hallucination_grader(context, good_answer))

In [ ]:
# Test 2: Hallucinated Answer
bad_answer = "The Eiffel Tower is located in Paris, France, and was built by aliens in 1889."
print("Hallucination Score:", hallucination_grader(context, bad_answer))

## 2. Answer Quality Grader


In [ ]:
def answer_quality_grader(question: str, answer: str):
    prompt = f"""
    You are a grader assessing whether an answer addresses / resolves a question.
    Give a binary score 'yes' or 'no'. Yes means that the answer resolves the question.
    
    Question: {question}
    
    Answer: {answer}
    
    Binary Score ("yes" or "no"):
    """
    response = llm.invoke(prompt)
    return response.content.strip().lower()

query = "How tall is the Eiffel Tower?"

In [ ]:
# Test 1: Useful Answer
good_answer = "The Eiffel Tower is 330 meters tall."
print("Quality Score:", answer_quality_grader(query, good_answer))

In [ ]:
# Test 2: Evasive Answer
bad_answer = "The Eiffel Tower is located in Paris, France and was built in 1889."
print("Quality Score:", answer_quality_grader(query, bad_answer))

## 3. Query Rewriter


In [ ]:
def rewrite_query(question: str):
    prompt = f"""
    You are a question re-writer that converts an input question to a better version that is optimized 
    for vectorstore retrieval. Look at the input and try to reason about the underlying semantic intent / meaning.
    
    Initial question: {question}
    
    Formulate an improved question:
    """
    response = llm.invoke(prompt)
    return response.content.strip()


In [ ]:
ambiguous_query = "iphone battery fast drain fix"
print("Original:", ambiguous_query)
print("Rewritten:", rewrite_query(ambiguous_query))